# Tokatrons — BART Plan-Guided Inference (CLEF 2026 SimpleText)


## Description
This notebook runs BART (facebook/bart-large) fine-tuned with plan-guided decoding
(`[REPHRASE]`, `[DELETE]`, `[SPLIT]`, `[MERGE]`, `[COPY]`) on the CLEF 2026 SimpleText
Task 11 test set (48,809 sentences). The model is loaded from a Drive checkpoint at
`models/bart-plan-guided`. Input data is JSON with `complex` sentence strings; output
is a CLEF-compatible submission JSON with predicted simplifications.


## Hyperparameters
- Batch size: 8 (TPU v5e-1), 16 (CPU fallback)
- Max input length: 128 tokens
- Max output length: 64 tokens
- Decoding: greedy (num_beams=1, do_sample=False)
- Checkpoint: every 100 sentences to Drive
- XLA compilation warmup: first batch freezes ~2-5 min on TPU


In [ ]:
# ============================================================
# BART v2 — TPU v5e-1 optimized | 48809 sentences
# Runtime → Change runtime type → TPU v5e-1
# ============================================================
import subprocess
subprocess.run(['pip', 'install', 'transformers', 'sentencepiece', 'cloud-tpu-client', '-q'])

import os, json, zipfile, time, torch
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'   # suppress warnings
os.environ['TOKENIZERS_PARALLELISM'] = 'false'   # suppress tokenizer warning

from transformers import BartTokenizer, BartForConditionalGeneration
from google.colab import drive
drive.mount('/content/drive')

# ── TPU setup ──
try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    device = torch_xla.device()
    IS_TPU = True
    print(f"[OK] TPU v5e-1 detected: {device}", flush=True)
except ImportError:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    IS_TPU = False
    print(f"[WARN]  TPU not available, falling back to: {device}", flush=True)

# ── Load BART v2 ──
print("Loading tokenizer...", flush=True)
v2_path   = '/content/drive/MyDrive/SimpleText2025/models/bart-plan-guided'
tokenizer = BartTokenizer.from_pretrained(v2_path)

print("Loading model...", flush=True)
model = BartForConditionalGeneration.from_pretrained(v2_path)

# ── Clean generation config ──
model.generation_config.forced_bos_token_id = 0
model.generation_config.early_stopping      = None  # not valid for greedy
model.generation_config.length_penalty      = None  # not valid for greedy

special_tokens = ['[REPHRASE]','[DELETE]','[SPLIT]','[MERGE]','[COPY]']
tokenizer.add_tokens(special_tokens)
model.resize_token_embeddings(len(tokenizer))

print("Moving model to device...", flush=True)
model = model.to(device)
model.eval()
print(f"[OK] BART v2 loaded on {device}", flush=True)

# ── Paths ──
DATA_PATH = '/content/drive/MyDrive/input/sentences_src.json'
OUT_DIR   = '/content/drive/MyDrive/SimpleText2025/outputs'
OUT_PATH  = f'{OUT_DIR}/tokatrons_task11_BART_2026.json'
ZIP_PATH  = OUT_PATH.replace('.json', '.zip')
CKPT_PATH = f'{OUT_DIR}/bart_2026_checkpoint.json'
os.makedirs(OUT_DIR, exist_ok=True)

print("Loading source data...", flush=True)
with open(DATA_PATH) as f:
    src_data = json.load(f)
total     = len(src_data)
sentences = [d['complex'] for d in src_data]
print(f"[OK] Loaded {total} sentences", flush=True)

# ── Load checkpoint ──
if os.path.exists(CKPT_PATH):
    with open(CKPT_PATH) as f:
        ckpt = json.load(f)
    preds      = ckpt['preds']
    start_from = len(preds)
    print(f"[OK] Resuming from {start_from}/{total}", flush=True)
    print(f"   Delete {CKPT_PATH} to start fresh", flush=True)
else:
    preds      = []
    start_from = 0
    print("Starting fresh", flush=True)

# ── v5e-1 optimized settings ──
BATCH_SIZE  = 8 if IS_TPU else 16
MAX_IN_LEN  = 128
MAX_OUT_LEN = 64
CKPT_EVERY  = 100

start_time = time.time()

print(f"\n[START] Starting from sentence {start_from}, {total - start_from} remaining...", flush=True)
print(f"   Batch size : {BATCH_SIZE}", flush=True)
print(f"   Max in len : {MAX_IN_LEN}", flush=True)
print(f"   Max out len: {MAX_OUT_LEN}", flush=True)
print(f"   Device     : {device}", flush=True)
print(f"   Decoding   : greedy (num_beams=1, fastest)", flush=True)
if IS_TPU:
    print(f"\n   [WARN]  First batch freezes ~2-5 min for XLA compilation — this is normal!", flush=True)
    print(f"   [FAST] After warmup expect ~80-120 sent/sec\n", flush=True)

for i in range(start_from, total, BATCH_SIZE):
    batch  = sentences[i:i+BATCH_SIZE]
    guided = ['[REPHRASE] ' + s for s in batch]

    # Step 1 — Tokenize
    print(f"  [TOK] [{time.strftime('%H:%M:%S')}] Tokenizing  {i}→{i+len(batch)}...", flush=True)
    enc = tokenizer(
        guided,
        return_tensors='pt',
        max_length=MAX_IN_LEN,
        truncation=True,
        padding='max_length'    # static shape — TPU compiles graph once only
    )
    enc = {k: v.to(device) for k, v in enc.items()}

    # Step 2 — Generate
    print(f"  [GEN]  [{time.strftime('%H:%M:%S')}] Generating  {i}→{i+len(batch)}...", flush=True)
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=MAX_OUT_LEN,   # use max_new_tokens instead of max_length
            num_beams=1,
            do_sample=False,
            forced_bos_token_id=0,
            early_stopping=False,         # explicitly disable
            length_penalty=1.0,           # explicitly neutral
        )

    # Step 3 — mark_step before pulling to CPU
    if IS_TPU:
        print(f"  [SYNC] [{time.strftime('%H:%M:%S')}] mark_step   {i}→{i+len(batch)}...", flush=True)
        xm.mark_step()

    # Step 4 — Decode
    print(f"  [DEC] [{time.strftime('%H:%M:%S')}] Decoding    {i}→{i+len(batch)}...", flush=True)
    out_cpu = out.cpu()
    decoded = tokenizer.batch_decode(out_cpu, skip_special_tokens=True)
    preds.extend(decoded)

    # Step 5 — Print every batch
    now     = time.time()
    elapsed = now - start_time
    done    = len(preds) - start_from
    speed   = done / elapsed if elapsed > 0 else 0
    eta     = (total - len(preds)) / speed if speed > 0 else 0

    print(f"  [OK] [{time.strftime('%H:%M:%S')}] {len(preds)}/{total} | "
          f"{speed:.1f} sent/sec | "
          f"Elapsed: {elapsed/60:.1f} min | "
          f"ETA: {eta/60:.1f} min", flush=True)

    # Step 6 — Checkpoint every 100 sentences
    if len(preds) % CKPT_EVERY < BATCH_SIZE:
        if IS_TPU:
            xm.mark_step()
        with open(CKPT_PATH, 'w') as f:
            json.dump({'preds': preds}, f)
        print(f"  [SAVE] Checkpoint saved at {len(preds)}/{total} [{time.strftime('%H:%M:%S')}]", flush=True)

# ── Final sync + save ──
if IS_TPU:
    xm.mark_step()

with open(CKPT_PATH, 'w') as f:
    json.dump({'preds': preds}, f)

elapsed_total = time.time() - start_time
print(f"\n[OK] All {len(preds)} predictions done in {elapsed_total/60:.1f} min", flush=True)

# ── Build submission ──
print("\nBuilding submission file...", flush=True)
submission = []
for i, (entry, pred) in enumerate(zip(src_data, preds)):
    submission.append({
        "pair_id":    str(entry['pair_id']),
        "source":     entry.get('source', 'Cochrane-auto 2026'),
        "language":   entry.get('language', 'en'),
        "para_id":    int(entry['para_id']),
        "sent_id":    int(entry['sent_id']),
        "complex":    entry['complex'],
        "prediction": pred if pred.strip() != "" else entry['complex'],
        "run_id":     "tokatrons_task11_BART_2026"
    })

with open(OUT_PATH, 'w') as f:
    json.dump(submission, f, indent=2)
with zipfile.ZipFile(ZIP_PATH, 'w') as zf:
    zf.write(OUT_PATH, arcname='tokatrons_task11_BART_2026.json')

identical = sum(1 for s in submission
                if s['prediction'].strip() == s['complex'].strip())
print(f"\n[OK] Submission saved: {ZIP_PATH}", flush=True)
print(f"[STAT] Total: {len(submission)} | Changed: {len(submission)-identical} | Identical: {identical}", flush=True)
print(f"[TIME]  Total time: {elapsed_total/60:.1f} min", flush=True)
print(f"\nSample:", flush=True)
for entry in submission[:2]:
    print(json.dumps(entry, indent=4), flush=True)

if os.path.exists(CKPT_PATH):
    os.remove(CKPT_PATH)
    print("\n[OK] Checkpoint cleaned up", flush=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[WARN]  TPU not available, falling back to: cpu
Loading tokenizer...
Loading model...


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Moving model to device...
[OK] BART v2 loaded on cpu
Loading source data...
[OK] Loaded 48809 sentences
[OK] Resuming from 20704/48809
   Delete /content/drive/MyDrive/SimpleText2025/outputs/bart_2026_checkpoint.json to start fresh

[START] Starting from sentence 20704, 28105 remaining...
   Batch size : 16
   Max in len : 128
   Max out len: 64
   Device     : cpu
   Decoding   : greedy (num_beams=1, fastest)
  [TOK] [07:13:27] Tokenizing  20704→20720...
  [GEN]  [07:13:28] Generating  20704→20720...
  [DEC] [07:14:22] Decoding    20704→20720...
  [OK] [07:14:22] 20720/48809 | 0.3 sent/sec | Elapsed: 0.9 min | ETA: 1590.7 min
  [TOK] [07:14:22] Tokenizing  20720→20736...
  [GEN]  [07:14:22] Generating  20720→20736...
  [DEC] [07:14:55] Decoding    20720→20736...
  [OK] [07:14:55] 20736/48809 | 0.4 sent/sec | Elapsed: 1.5 min | ETA: 1277.8 min
  [TOK] [07:14:55] Tokenizing  20736→20752...
  [GEN]  [07:14:55] Generating  20736→20752...
  [DEC] [07:15:28] Decoding    20736→20752...
  [OK

KeyboardInterrupt: 

In [ ]:
# Run this in a NEW cell while main cell is running
import subprocess
result = subprocess.run(['ps', 'aux'], capture_output=True, text=True)
xla_procs = [l for l in result.stdout.split('\n') if 'xla' in l.lower() or 'python' in l.lower()]
for p in xla_procs:
    print(p)